In [23]:
from pathlib import Path
import geopandas as gpd
import cdsapi
import xarray as xr
import pandas as pd
import numpy as np
import regionmask


01_define_study_area

In [ ]:
PROJECT_DIR = Path.cwd().parent

boundary_file = PROJECT_DIR / "data" / "raw" / "gadm41_VNM.json"
output_dir = PROJECT_DIR / "data" / "processed"

print("Project directory:", PROJECT_DIR)
print("Boundary file exists:", boundary_file.exists())
print("Boundary file:", boundary_file)

Project directory: /home/trang/Drought_prediction
Boundary file exists: True
Boundary file: /home/trang/Drought_prediction/data/raw/gadm41_VNM.json


02_download_era5

In [ ]:
# ==========================================
# 1. CẤU HÌNH ĐƯỜNG DẪN
# ==========================================
# Notebook đang nằm trong thư mục scripts/
PROJECT_DIR = Path.cwd().parent

RAW_DIR = PROJECT_DIR / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = RAW_DIR / "era5_land_monthly_2000_2025.nc"

print("Project directory:", PROJECT_DIR)
print("Output file:", OUTPUT_FILE)

# ==========================================
# 2. CẤU HÌNH DATASET ERA5-LAND
# ==========================================
dataset = "reanalysis-era5-land-monthly-means"

request = {
    "product_type": ["monthly_averaged_reanalysis"],

    "variable": [
        "2m_dewpoint_temperature",
        "2m_temperature",
        "volumetric_soil_water_layer_1",
        "volumetric_soil_water_layer_2",
        "volumetric_soil_water_layer_3",
        "volumetric_soil_water_layer_4",
        "potential_evaporation",
        "total_precipitation",
    ],

    "year": [str(year) for year in range(2000, 2026)],
    "month": [f"{month:02d}" for month in range(1, 13)],
    "time": ["00:00"],

    # [North, West, South, East]
    "area": [11.1, 103.3, 8.0, 107.0],

    "data_format": "netcdf",
    "download_format": "unarchived",
}

# ==========================================
# 3. TẢI DỮ LIỆU
# ==========================================
if OUTPUT_FILE.exists():
    print(f"File đã tồn tại: {OUTPUT_FILE}")
    print("Không tải lại để tránh ghi đè dữ liệu hiện có.")
else:
    client = cdsapi.Client()

    print("Bắt đầu gửi yêu cầu tải ERA5-Land...")
    client.retrieve(dataset, request, str(OUTPUT_FILE))

    print(f"\nTải hoàn tất: {OUTPUT_FILE}")


Project directory: /home/trang/Drought_prediction
Output file: /home/trang/Drought_prediction/data/raw/era5_land_monthly_2000_2025.nc
File đã tồn tại: /home/trang/Drought_prediction/data/raw/era5_land_monthly_2000_2025.nc
Không tải lại để tránh ghi đè dữ liệu hiện có.


In [ ]:
# Các tỉnh/thành ĐBSCL theo ranh giới hành chính trong GADM này
mekong_provinces = [
    "AnGiang",
    "BạcLiêu",
    "BếnTre",
    "CàMau",
    "CầnThơ",
    "ĐồngTháp",
    "HậuGiang",
    "KiênGiang",
    "LongAn",
    "SócTrăng",
    "TiềnGiang",
    "TràVinh",
    "VĩnhLong",
]

gdf = gpd.read_file(boundary_file)

# Xem tên tỉnh thực tế có khớp danh sách không
available = set(gdf["NAME_1"].dropna())
selected = [name for name in mekong_provinces if name in available]
missing = [name for name in mekong_provinces if name not in available]

print("Tỉnh tìm thấy:", selected)
print("Tên chưa khớp:", missing)

if missing:
    raise ValueError(
        "Có tên tỉnh chưa khớp với dữ liệu GADM. "
        "Hãy kiểm tra NAME_1 trước khi tiếp tục."
    )

# Chọn và hợp nhất các polygon tỉnh
mekong = gdf[gdf["NAME_1"].isin(selected)].copy()
mekong = mekong.to_crs("EPSG:4326")

region = mekong[["geometry"]].dissolve()
region["region"] = "Mekong_Delta"

output_file = output_dir / "mekong_delta_boundary.geojson"
region.to_file(output_file, driver="GeoJSON")

print("Số tỉnh được chọn:", len(selected))
print("Đã lưu:", output_file)
print("Bounding box:", region.total_bounds)

Tỉnh tìm thấy: ['AnGiang', 'BạcLiêu', 'BếnTre', 'CàMau', 'CầnThơ', 'ĐồngTháp', 'HậuGiang', 'KiênGiang', 'LongAn', 'SócTrăng', 'TiềnGiang', 'TràVinh', 'VĩnhLong']
Tên chưa khớp: []
Số tỉnh được chọn: 13
Đã lưu: /home/trang/Drought_prediction/data/processed/mekong_delta_boundary.geojson
Bounding box: [103.4588   8.3814 106.8225  11.0323]


In [ ]:
PROJECT_DIR = Path.cwd().parent
RAW_DIR = PROJECT_DIR / "data" / "raw"
OUTPUT_FILE = RAW_DIR / "era5_land_monthly_2000_2025.nc"

print("File tồn tại:", OUTPUT_FILE.exists())

if OUTPUT_FILE.exists():
    print(f"Dung lượng: {OUTPUT_FILE.stat().st_size / (1024**2):.2f} MB")
    print("Đường dẫn:", OUTPUT_FILE)

File tồn tại: True
Dung lượng: 3.20 MB
Đường dẫn: /home/trang/Drought_prediction/data/raw/era5_land_monthly_2000_2025.nc


In [ ]:

OUTPUT_FILE = PROJECT_DIR / "data" / "raw" / "era5_land_monthly_2000_2025.nc"

# Mở file NetCDF hiện có, không tải lại
with xr.open_dataset(OUTPUT_FILE) as ds:
    print("=== 1. CẤU TRÚC DATASET ===")
    print(ds)

    print("\n=== 2. CÁC BIẾN KHÍ TƯỢNG ===")
    for var in ds.data_vars:
        print(f"{var}: {ds[var].attrs.get('long_name', 'N/A')}")
        print(f"    Units: {ds[var].attrs.get('units', 'N/A')}")

    print("\n=== 3. KÍCH THƯỚC DỮ LIỆU ===")
    print(ds.sizes)

    print("\n=== 4. KHOẢNG THỜI GIAN ===")
    time = ds["valid_time"]
    print("Bắt đầu:", pd.Timestamp(time.min().values))
    print("Kết thúc:", pd.Timestamp(time.max().values))
    print("Số mốc thời gian:", time.size)

    print("\n=== 5. PHẠM VI TỌA ĐỘ ===")
    print(
        "Latitude:",
        float(ds["latitude"].min()),
        "→",
        float(ds["latitude"].max())
    )
    print(
        "Longitude:",
        float(ds["longitude"].min()),
        "→",
        float(ds["longitude"].max())
    )


=== 1. CẤU TRÚC DATASET ===
<xarray.Dataset> Size: 12MB
Dimensions:     (valid_time: 312, latitude: 32, longitude: 38)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 2kB 2000-01-01 ... 2025-12-01
    expver      (valid_time) <U4 5kB ...
  * latitude    (latitude) float64 256B 11.1 11.0 10.9 10.8 ... 8.3 8.2 8.1 8.0
  * longitude   (longitude) float64 304B 103.3 103.4 103.5 ... 106.8 106.9 107.0
    number      int64 8B ...
Data variables:
    d2m         (valid_time, latitude, longitude) float32 2MB ...
    t2m         (valid_time, latitude, longitude) float32 2MB ...
    swvl1       (valid_time, latitude, longitude) float32 2MB ...
    swvl2       (valid_time, latitude, longitude) float32 2MB ...
    swvl3       (valid_time, latitude, longitude) float32 2MB ...
    swvl4       (valid_time, latitude, longitude) float32 2MB ...
    pev         (valid_time, latitude, longitude) float32 2MB ...
    tp          (valid_time, latitude, longitude) float32 2MB ...
Attributes:
    GRI

In [ ]:



BOUNDARY_FILE = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "mekong_delta_boundary.geojson"
)

assert BOUNDARY_FILE.exists(), (
    f"Chưa tìm thấy file ranh giới: {BOUNDARY_FILE}"
)

region = gpd.read_file(BOUNDARY_FILE)

print("=== THÔNG TIN RANH GIỚI ===")
print("Số polygon:", len(region))
print("CRS:", region.crs)
print("Geometry type:", region.geometry.geom_type.tolist())
print("Geometry hợp lệ:", region.geometry.is_valid.all())
print("Geometry rỗng:", region.geometry.is_empty.any())
print("Bounding box [min_lon, min_lat, max_lon, max_lat]:")
print(region.total_bounds)


=== THÔNG TIN RANH GIỚI ===
Số polygon: 1
CRS: EPSG:4326
Geometry type: ['MultiPolygon']
Geometry hợp lệ: True
Geometry rỗng: False
Bounding box [min_lon, min_lat, max_lon, max_lat]:
[103.4588   8.3814 106.8225  11.0323]


In [ ]:

RAW_FILE = (
    PROJECT_DIR
    / "data"
    / "raw"
    / "era5_land_monthly_2000_2025.nc"
)

CLIPPED_FILE = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "era5_land_mekong_delta_2000_2025.nc"
)

region = gpd.read_file(BOUNDARY_FILE).to_crs("EPSG:4326")

with xr.open_dataset(RAW_FILE) as ds:
    # Tạo mặt nạ từ polygon ĐBSCL trên lưới ERA5-Land
    mask = regionmask.mask_geopandas(
        region,
        ds["longitude"],
        ds["latitude"]
    )

    # Giữ ô lưới thuộc polygon, các ô ngoài vùng thành NaN
    clipped = ds.where(mask == 0)

    # Xác nhận có dữ liệu trong vùng nghiên cứu
    # Đếm số ô lưới có dữ liệu tại tháng đầu tiên
    valid_cells = int(
        clipped["t2m"]
        .isel(valid_time=0)
        .notnull()
        .sum()
        .item()
    )

    print("Số ô lưới có dữ liệu tại tháng đầu:", valid_cells)

    if valid_cells == 0:
        raise ValueError(
            "Không có ô lưới nào được giữ lại. "
            "Cần kiểm tra polygon và tọa độ."
        )

    clipped.to_netcdf(CLIPPED_FILE)

print("Đã lưu dữ liệu cắt tại:", CLIPPED_FILE)


Số ô lưới có dữ liệu tại tháng đầu: 326
Đã lưu dữ liệu cắt tại: /home/trang/Drought_prediction/data/processed/era5_land_mekong_delta_2000_2025.nc


In [ ]:
CLIPPED_FILE = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "era5_land_mekong_delta_2000_2025.nc"
)

with xr.open_dataset(CLIPPED_FILE) as ds:
    print("=== CẤU TRÚC DATASET SAU KHI CẮT ===")
    print(ds)

    print("\n=== KIỂM TRA THỜI GIAN ===")
    print("Số tháng:", ds.sizes["valid_time"])
    print("Bắt đầu:", str(ds["valid_time"].values[0])[:10])
    print("Kết thúc:", str(ds["valid_time"].values[-1])[:10])

    print("\n=== KIỂM TRA SỐ Ô LƯỚI ===")
    valid_mask = ds["t2m"].isel(valid_time=0).notnull()
    print("Số ô có dữ liệu:", int(valid_mask.sum().item()))
    print("Tổng số vị trí trong lưới:",
          valid_mask.sizes["latitude"] * valid_mask.sizes["longitude"])

    print("\n=== KIỂM TRA MISSING VALUES ===")
    for var in ds.data_vars:
        missing = int(ds[var].isnull().sum().item())
        total = ds[var].size
        print(f"{var}: {missing:,}/{total:,} "
              f"({missing / total:.2%})")


=== CẤU TRÚC DATASET SAU KHI CẮT ===
<xarray.Dataset> Size: 12MB
Dimensions:     (valid_time: 312, latitude: 32, longitude: 38)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 2kB 2000-01-01 ... 2025-12-01
    expver      (valid_time) <U4 5kB ...
  * latitude    (latitude) float64 256B 11.1 11.0 10.9 10.8 ... 8.3 8.2 8.1 8.0
  * longitude   (longitude) float64 304B 103.3 103.4 103.5 ... 106.8 106.9 107.0
    number      int64 8B ...
Data variables:
    d2m         (valid_time, latitude, longitude) float32 2MB ...
    t2m         (valid_time, latitude, longitude) float32 2MB ...
    swvl1       (valid_time, latitude, longitude) float32 2MB ...
    swvl2       (valid_time, latitude, longitude) float32 2MB ...
    swvl3       (valid_time, latitude, longitude) float32 2MB ...
    swvl4       (valid_time, latitude, longitude) float32 2MB ...
    pev         (valid_time, latitude, longitude) float32 2MB ...
    tp          (valid_time, latitude, longitude) float32 2MB ...
Attributes

In [ ]:
RAW_FILE = PROJECT_DIR / "data" / "raw" / "era5_land_monthly_2000_2025.nc"
CLIPPED_FILE = PROJECT_DIR / "data" / "processed" / "era5_land_mekong_delta_2000_2025.nc"

with xr.open_dataset(RAW_FILE) as raw, xr.open_dataset(CLIPPED_FILE) as clipped:
    # Ô có dữ liệu tại tháng đầu tiên trong file gốc
    raw_mask = raw["t2m"].isel(valid_time=0).notnull()

    # Ô có dữ liệu tại tháng đầu tiên sau khi cắt
    clipped_mask = clipped["t2m"].isel(valid_time=0).notnull()

    print("=== SO SÁNH LƯỚI ===")
    print("Kích thước gốc:", raw["t2m"].isel(valid_time=0).shape)
    print("Kích thước sau cắt:", clipped["t2m"].isel(valid_time=0).shape)
    print("Số ô có dữ liệu ở file gốc:", int(raw_mask.sum().item()))
    print("Số ô có dữ liệu sau cắt:", int(clipped_mask.sum().item()))
    print("Số ô bị loại:", int((raw_mask & ~clipped_mask).sum().item()))

    print("\n=== MISSING VALUES Ở THÁNG ĐẦU ===")
    for var in raw.data_vars:
        before = int(raw[var].isel(valid_time=0).isnull().sum().item())
        after = int(clipped[var].isel(valid_time=0).isnull().sum().item())
        print(f"{var}: gốc={before}, sau cắt={after}")


=== SO SÁNH LƯỚI ===
Kích thước gốc: (32, 38)
Kích thước sau cắt: (32, 38)
Số ô có dữ liệu ở file gốc: 494
Số ô có dữ liệu sau cắt: 326
Số ô bị loại: 168

=== MISSING VALUES Ở THÁNG ĐẦU ===
d2m: gốc=722, sau cắt=890
t2m: gốc=722, sau cắt=890
swvl1: gốc=722, sau cắt=890
swvl2: gốc=722, sau cắt=890
swvl3: gốc=722, sau cắt=890
swvl4: gốc=722, sau cắt=890
pev: gốc=722, sau cắt=890
tp: gốc=722, sau cắt=890


In [ ]:

CLIPPED_FILE = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "era5_land_mekong_delta_2000_2025.nc"
)

with xr.open_dataset(CLIPPED_FILE) as ds:
    rows = []

    for var in ds.data_vars:
        da = ds[var]
        missing = int(da.isnull().sum().item())
        total = da.size

        rows.append({
            "variable": var,
            "units": da.attrs.get("units", "N/A"),
            "missing_count": missing,
            "total_count": total,
            "missing_percent": round(missing / total * 100, 2),
        })

    quality_report = pd.DataFrame(rows)
    print(quality_report.to_string(index=False))


variable      units  missing_count  total_count  missing_percent
     d2m          K         277680       379392            73.19
     t2m          K         277680       379392            73.19
   swvl1 m**3 m**-3         277680       379392            73.19
   swvl2 m**3 m**-3         277680       379392            73.19
   swvl3 m**3 m**-3         277680       379392            73.19
   swvl4 m**3 m**-3         277680       379392            73.19
     pev          m         277680       379392            73.19
      tp          m         277680       379392            73.19


In [ ]:
CLIPPED_FILE = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "era5_land_mekong_delta_2000_2025.nc"
)

CSV_FILE = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "era5_land_mekong_delta_2000_2025.csv"
)

with xr.open_dataset(CLIPPED_FILE) as ds:
    # Chỉ giữ các tọa độ có dữ liệu t2m ở ít nhất một tháng
    spatial_mask = ds["t2m"].notnull().any(dim="valid_time")
    ds_region = ds.where(spatial_mask, drop=True)

    # Chuyển dữ liệu không gian-thời gian thành bảng
    df = ds_region.to_dataframe().reset_index()

    # Loại các hàng không thuộc polygon
    df = df.dropna(subset=["t2m"])

    # Chỉ giữ các cột cần thiết
    columns = [
        "valid_time",
        "latitude",
        "longitude",
        "d2m",
        "t2m",
        "swvl1",
        "swvl2",
        "swvl3",
        "swvl4",
        "pev",
        "tp",
    ]
    df = df[columns].rename(columns={"valid_time": "date"})

    df.to_csv(CSV_FILE, index=False)

print("Đã lưu CSV:", CSV_FILE)
print("Số dòng:", len(df))
print("Số cột:", len(df.columns))
print(df.head())


Đã lưu CSV: /home/trang/Drought_prediction/data/processed/era5_land_mekong_delta_2000_2025.csv
Số dòng: 101712
Số cột: 11
         date  latitude  longitude         d2m         t2m     swvl1  \
13 2000-01-01      11.0      105.7  293.165161  300.119019  0.303757   
19 2000-01-01      11.0      106.3  293.528442  299.745972  0.309219   
32 2000-01-01      10.9      105.1  293.647583  299.824097  0.178589   
33 2000-01-01      10.9      105.2  293.792114  299.802612  0.198517   
35 2000-01-01      10.9      105.4  293.696411  299.751831  0.202850   

       swvl2     swvl3     swvl4       pev        tp  
13  0.305405  0.366119  0.474777 -0.007460  0.000670  
19  0.306824  0.364578  0.471451 -0.007120  0.000832  
32  0.198395  0.271515  0.382568 -0.007319  0.000600  
33  0.215652  0.288864  0.384689 -0.007280  0.000649  
35  0.218765  0.296021  0.384338 -0.006971  0.000787  
